## 1️⃣ Setup Environment

In [ ]:
# @title 1.1 Check GPU Type
# @markdown Kiểm tra GPU để chọn mode phù hợp

import subprocess
import torch

def check_gpu():
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', 
                           '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"🖥️ GPU Info: {gpu_info}")
    
    if "A100" in gpu_info:
        print("\n✅ A100 detected - FULL MODE available (40GB VRAM)")
        return "full", 40
    elif "V100" in gpu_info:
        print("\n✅ V100 detected - FULL MODE available (16-32GB VRAM)")
        return "full", 16
    elif "T4" in gpu_info:
        print("\n⚠️ T4 detected - LITE MODE recommended (16GB VRAM)")
        print("   Full pipeline needs 18-20GB, using 4-bit quantization")
        return "lite", 16
    elif "P100" in gpu_info:
        print("\n⚠️ P100 detected - LITE MODE recommended (16GB VRAM)")
        return "lite", 16
    else:
        print("\n❓ Unknown GPU - Using DEMO MODE")
        return "demo", 8

MODE, VRAM_GB = check_gpu()
print(f"\n📊 Selected Mode: {MODE.upper()}")
print(f"📊 Available VRAM: {VRAM_GB}GB")

In [ ]:
# @title 1.2 Install Dependencies
# @markdown Cài đặt các thư viện cần thiết

!pip install -q torch torchvision torchaudio
!pip install -q transformers==4.36.0 accelerate==0.25.0
!pip install -q bitsandbytes  # For 4-bit quantization
!pip install -q open_clip_torch==2.23.0
!pip install -q gradio==4.8.0
!pip install -q einops timm safetensors sentencepiece
!pip install -q pillow requests

print("\n✅ Dependencies installed!")

In [ ]:
# @title 1.3 Import Libraries

import torch
import open_clip
import gradio as gr
from PIL import Image
import requests
from io import BytesIO
import json
import numpy as np

# Check CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Using device: {device}")

if device == "cuda":
    print(f"🔧 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔧 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

## 2️⃣ Load Models

In [ ]:
# @title 2.1 Load BiomedCLIP (Triage + Gatekeeper)
# @markdown Model cho Visual Triage và Gatekeeper verification

print("📥 Loading BiomedCLIP...")

biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
biomedclip_model = biomedclip_model.to(device).eval()

print(f"✅ BiomedCLIP loaded!")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f}GB")

In [ ]:
# @title 2.2 Configuration
# @markdown Cấu hình cho TriMedAgent

TRIMED_CONFIG = {
    "triage_labels": [
        "Chest X-ray",
        "Brain MRI",
        "Abdominal CT",
        "Histopathology",
        "Ultrasound",
        "Dermoscopy",
        "Gross pathology",
        "Bone X-ray",
        "Lung CT",
        "Retinal fundus",
        "Mammography"
    ],
    "gatekeeper_prompts": {
        "positive": "Pathological finding, lesion, tumor, abnormality",
        "negative": "Normal tissue, healthy anatomy, background noise, blurry area"
    },
    "thresholds": {
        "triage_confidence": 0.5,
        "gatekeeper_confidence": 0.6
    },
    "conditional_execution": {
        "skip_detection_modalities": ["Histopathology", "Dermoscopy"],
        "tool_mapping": {
            "Chest X-ray": ["grounding_dino", "medsam"],
            "Brain MRI": ["grounding_dino", "medsam"],
            "Abdominal CT": ["grounding_dino", "medsam"],
            "Histopathology": ["biomedclip"],
            "default": ["grounding_dino"]
        }
    }
}

print("✅ Configuration loaded!")
print(f"   Triage labels: {len(TRIMED_CONFIG['triage_labels'])}")

## 3️⃣ TriMedAgent Class

In [ ]:
# @title 3.1 TriMedAgent Implementation

class TriMedAgent:
    """TriMedAgent - Intelligent Triage Pipeline for Medical Image Analysis"""
    
    def __init__(self, model, preprocess, tokenizer, config, device="cuda"):
        self.model = model
        self.preprocess = preprocess
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        
        # State
        self.last_triage_result = None
        self.conversation_history = []
        self.current_image = None
        self.session_active = False
        self.detected_boxes = []
        self.verified_boxes = []
    
    def _classify(self, image, labels, template="this is a photo of {}"):
        """Core classification using BiomedCLIP"""
        image_tensor = self.preprocess(image).unsqueeze(0).to(self.device)
        texts = self.tokenizer([template.format(l) for l in labels]).to(self.device)
        
        with torch.no_grad():
            image_features = self.model.encode_image(image_tensor)
            text_features = self.model.encode_text(texts)
            
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            
            probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            probs = probs.cpu().numpy()[0]
        
        all_preds = dict(zip(labels, probs.tolist()))
        top_idx = probs.argmax()
        
        return {
            "prediction": labels[top_idx],
            "confidence": float(probs[top_idx]),
            "all_predictions": all_preds
        }
    
    def run_triage(self, image, verbose=True):
        """STAGE 1: Visual Triage"""
        labels = self.config["triage_labels"]
        result = self._classify(image, labels, "this is a {}")
        
        tool_mapping = self.config.get("conditional_execution", {}).get("tool_mapping", {})
        recommended_tools = tool_mapping.get(result["prediction"], tool_mapping.get("default", ["grounding_dino"]))
        skip_detection = result["prediction"] in self.config.get("conditional_execution", {}).get("skip_detection_modalities", [])
        
        threshold = self.config["thresholds"]["triage_confidence"]
        if result["confidence"] >= threshold:
            context = f"[System: Image is {result['prediction']} ({result['confidence']:.0%}). Tools: {', '.join(recommended_tools)}]"
        else:
            context = f"[System: Unclear image type (guess: {result['prediction']}, {result['confidence']:.0%})]"
        
        triage_result = {
            "modality": result["prediction"],
            "confidence": result["confidence"],
            "all_predictions": result["all_predictions"],
            "recommended_tools": recommended_tools,
            "skip_detection": skip_detection,
            "context_prompt": context
        }
        
        self.last_triage_result = triage_result
        
        if verbose:
            print(f"\n🔬 TRIAGE RESULT:")
            print(f"   Modality: {result['prediction']}")
            print(f"   Confidence: {result['confidence']:.1%}")
            print(f"   Tools: {recommended_tools}")
        
        return triage_result
    
    def run_gatekeeper(self, image, boxes, target="abnormality", verbose=True):
        """STAGE 3: Gatekeeper Verification"""
        if not boxes:
            return {"verified_boxes": [], "rejected_indices": [], "details": []}
        
        w, h = image.size
        threshold = self.config["thresholds"]["gatekeeper_confidence"]
        
        pos_label = f"{self.config['gatekeeper_prompts']['positive']} of {target}"
        neg_label = self.config['gatekeeper_prompts']['negative']
        labels = [pos_label, neg_label]
        
        verified_boxes = []
        rejected_indices = []
        details = []
        
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box
            crop = image.crop((int(x1*w), int(y1*h), int(x2*w), int(y2*h)))
            result = self._classify(crop, labels, "this is {}")
            
            passed = result["prediction"] == pos_label and result["confidence"] >= threshold
            
            if passed:
                verified_boxes.append(box)
            else:
                rejected_indices.append(i)
            
            details.append({"box_index": i, "passed": passed, "confidence": result["confidence"]})
            
            if verbose:
                status = "✅" if passed else "❌"
                print(f"   Box {i}: {status} ({result['confidence']:.1%})")
        
        return {"verified_boxes": verified_boxes, "rejected_indices": rejected_indices, "details": details}
    
    def start_session(self, image=None):
        """Start new chat session"""
        self.conversation_history = []
        self.current_image = image
        self.last_triage_result = None
        self.detected_boxes = []
        self.verified_boxes = []
        self.session_active = True
        
        if image is not None:
            self.last_triage_result = self.run_triage(image, verbose=True)
        
        return self.last_triage_result
    
    def chat(self, message, image=None):
        """Main chat function"""
        if image is not None:
            self.current_image = image
            self.last_triage_result = self.run_triage(image, verbose=False)
        
        if not self.session_active:
            self.start_session(self.current_image)
        
        # Save to history
        self.conversation_history.append({"role": "user", "content": message})
        
        # Generate response based on intent
        response = self._generate_response(message)
        
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response
    
    def _generate_response(self, message):
        """Generate response based on intent detection"""
        msg_lower = message.lower()
        
        if self.current_image is None:
            return "👋 Please upload a medical image first to start analysis."
        
        triage = self.last_triage_result
        modality = triage['modality'] if triage else 'Unknown'
        confidence = triage['confidence'] if triage else 0
        
        # Detection intent
        if any(w in msg_lower for w in ["detect", "find", "locate", "where", "abnormal"]):
            # Simulate detection
            if "X-ray" in modality:
                self.detected_boxes = [[0.25, 0.3, 0.45, 0.7], [0.55, 0.3, 0.75, 0.7], [0.35, 0.25, 0.65, 0.45]]
                target = "lung opacity"
            else:
                self.detected_boxes = [[0.3, 0.3, 0.7, 0.7]]
                target = "abnormality"
            
            print("\n🛡️ GATEKEEPER VERIFICATION:")
            gate_result = self.run_gatekeeper(self.current_image, self.detected_boxes, target)
            self.verified_boxes = gate_result["verified_boxes"]
            
            return f"""🔍 **Detection Results for {modality}**

**Tool:** Grounding DINO + Gatekeeper

**Raw Detections:** {len(self.detected_boxes)} regions
**After Gatekeeper:** {len(self.verified_boxes)} verified

{'✅ Verified regions found' if self.verified_boxes else '✅ No significant abnormalities detected'}"""
        
        # Classification intent
        if any(w in msg_lower for w in ["what", "type", "classify", "kind"]):
            top5 = sorted(triage['all_predictions'].items(), key=lambda x: x[1], reverse=True)[:5]
            preds_str = "\n".join([f"  • {l}: {p:.1%}" for l, p in top5])
            return f"""📋 **Classification Results**

**Type:** {modality}
**Confidence:** {confidence:.1%}

**Top-5 Predictions:**
{preds_str}"""
        
        # Description intent
        if any(w in msg_lower for w in ["describe", "explain", "tell", "see"]):
            return f"""📝 **Image Analysis**

This appears to be a **{modality}** image (confidence: {confidence:.1%}).

**Recommended Tools:** {', '.join(triage['recommended_tools'])}

You can ask me to:
- Detect abnormalities
- Segment regions of interest
- Provide more classification details"""
        
        # Default
        return f"""I'm analyzing a **{modality}** image.

How can I help? Try asking:
- "What type of image is this?"
- "Detect any abnormalities"
- "Describe what you see"""

# Initialize agent
agent = TriMedAgent(biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer, TRIMED_CONFIG, device)
print("\n✅ TriMedAgent initialized!")

## 4️⃣ Test with Sample Image

In [ ]:
# @title 4.1 Load Sample Image
# @markdown Tải ảnh test từ internet

# Sample medical images from public sources
SAMPLE_IMAGES = {
    "chest_xray": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_X-ray_in_influenza_and_Haemophilus_influenzae_-_annotated.jpg/800px-Chest_X-ray_in_influenza_and_Haemophilus_influenzae_-_annotated.jpg",
    "brain_mri": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/MRI_Head_Brain_Normal.jpg/800px-MRI_Head_Brain_Normal.jpg",
    "ct_scan": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/be/Computed_tomography_of_brain_of_Hashimoto%27s_encephalopathy_patient.jpg/800px-Computed_tomography_of_brain_of_Hashimoto%27s_encephalopathy_patient.jpg"
}

# Select image
selected = "chest_xray"  # @param ["chest_xray", "brain_mri", "ct_scan"]

print(f"📥 Loading {selected}...")
response = requests.get(SAMPLE_IMAGES[selected])
test_image = Image.open(BytesIO(response.content)).convert("RGB")

# Display
print(f"✅ Image loaded: {test_image.size}")
test_image.resize((400, 400))

In [ ]:
# @title 4.2 Test Triage Pipeline

print("="*60)
print("🔬 RUNNING TRIAGE PIPELINE")
print("="*60)

triage_result = agent.run_triage(test_image, verbose=True)

print("\n" + "="*60)
print("📊 TOP-5 PREDICTIONS")
print("="*60)
sorted_preds = sorted(triage_result['all_predictions'].items(), key=lambda x: x[1], reverse=True)[:5]
for i, (label, prob) in enumerate(sorted_preds, 1):
    bar = "█" * int(prob * 30)
    print(f"{i}. {label}: {prob:.1%} {bar}")

In [ ]:
# @title 4.3 Test Chatbot Conversation

print("="*60)
print("💬 CHATBOT CONVERSATION TEST")
print("="*60)

# Start session
agent.start_session(test_image)

# Test questions
questions = [
    "What type of image is this?",
    "Can you detect any abnormalities?",
    "Describe what you see"
]

for q in questions:
    print(f"\n👤 User: {q}")
    response = agent.chat(q)
    print(f"\n🤖 Agent:\n{response}")
    print("-"*40)

## 5️⃣ Gradio Chatbot UI

In [ ]:
# @title 5.1 Create Gradio Interface

def upload_image(image):
    """Handle image upload"""
    if image is None:
        return "Please upload an image.", []
    
    triage = agent.start_session(image)
    
    welcome = f"""🖼️ **Image Uploaded!**

**Triage Results:**
- Type: {triage['modality']}
- Confidence: {triage['confidence']:.1%}
- Tools: {', '.join(triage['recommended_tools'])}

Ask me anything about this image!"""
    
    return welcome, [[None, welcome]]

def respond(message, history):
    """Handle chat messages"""
    if not message:
        return "", history
    
    response = agent.chat(message)
    history.append([message, response])
    return "", history

def clear_chat():
    """Clear session"""
    agent.session_active = False
    agent.current_image = None
    return None, [], "Session cleared."

# Build UI
with gr.Blocks(title="TriMedAgent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏥 TriMedAgent Medical Chatbot
    
    **3-Stage Pipeline:** Perception → Reasoning → Gatekeeping
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="📷 Medical Image", type="pil", height=300)
            upload_btn = gr.Button("🚀 Analyze", variant="primary")
            status = gr.Textbox(label="Status", value="Upload image to start")
        
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="💬 Chat", height=400)
            with gr.Row():
                msg = gr.Textbox(label="Message", placeholder="Ask about the image...", scale=4)
                send_btn = gr.Button("Send", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear")
    
    # Events
    upload_btn.click(upload_image, [image_input], [status, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(clear_chat, [], [image_input, chatbot, status])

print("✅ Gradio UI created!")

In [ ]:
# @title 5.2 Launch Chatbot 🚀
# @markdown Chạy cell này để mở chatbot!

SHARE = True  # @param {type:"boolean"}
# @markdown `SHARE=True` sẽ tạo public link (72h)

demo.launch(share=SHARE, debug=True)

---

## 📝 Notes

### GPU Requirements:
- **Demo Mode (BiomedCLIP only):** ~1GB VRAM ✅
- **Full Pipeline (all models):** ~18-20GB VRAM

### This Demo Includes:
- ✅ Visual Triage (Stage 1)
- ✅ Context Injection (Stage 2)
- ✅ Gatekeeper Simulation (Stage 3)
- ⚠️ Detection/Segmentation simulated (need full pipeline)

### For Full Pipeline:
Use Colab Pro with A100 or rent GPU from RunPod/Vast.ai